In [21]:
pip install pandas requests beautifulsoup4 lxml soccerdata kagglehub

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import requests
import json
import random
import time
from collections import defaultdict
from bs4 import BeautifulSoup

# ── DATA SOURCE 1: Historical international results ────────────────────────
# martj42's dataset — 49,000+ international matches from 1872 to present.
# Free, public, no API key. This replaces the hardcoded BASE_ELO entirely.
RESULTS_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

print("Fetching historical international match data...")
historical_df = pd.read_csv(RESULTS_URL)
historical_df['date'] = pd.to_datetime(historical_df['date'])

print(f"  ✓ {len(historical_df):,} total international matches")
print(f"  ✓ Date range: {historical_df['date'].min().year} → {historical_df['date'].max().year}")
print(f"  Columns: {list(historical_df.columns)}")

# ── DATA SOURCE 2: 2026 World Cup match results ────────────────────────────
# openfootball — live WC 2026 match data. Updates as matches are played.
# No API key. Just a URL. Re-run this cell to get latest results.
WC_URL = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"

print("\nFetching 2026 World Cup match data...")
wc_data    = requests.get(WC_URL).json()
wc_matches = wc_data['matches']
completed  = [m for m in wc_matches if m.get('score')]

print(f"  ✓ {len(wc_matches)} total WC matches | {len(completed)} completed so far")
print(f"\nExample completed match:")
print(json.dumps(completed[0], indent=2))

Fetching historical international match data...
  ✓ 49,477 total international matches
  ✓ Date range: 1872 → 2026
  Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Fetching 2026 World Cup match data...
  ✓ 104 total WC matches | 56 completed so far

Example completed match:
{
  "round": "Matchday 1",
  "date": "2026-06-11",
  "time": "13:00 UTC-6",
  "team1": "Mexico",
  "team2": "South Africa",
  "score": {
    "ft": [
      2,
      0
    ],
    "ht": [
      1,
      0
    ]
  },
  "goals1": [
    {
      "name": "Juli\u00e1n Qui\u00f1ones",
      "minute": "9"
    },
    {
      "name": "Ra\u00fal Jim\u00e9nez",
      "minute": "67"
    }
  ],
  "goals2": [],
  "group": "Group A",
  "ground": "Mexico City"
}


In [23]:
# ── CELL 1B: FETCH xG DATA ────────────────────────────────────────────────
# Direct pd.read_html() scrape of FBRef schedule page.
# No browser needed. No selenium. No chromedriver.
# Sends a simple HTTP request with a browser User-Agent header —
# fast, reliable, 2-3 seconds total.
#
# soccerdata was tested but abandoned: FBRef's bot detection hangs
# headless Chrome sessions indefinitely (500+ seconds with no output).
# pd.read_html() bypasses that entirely.

import time

FBREF_URL = "https://fbref.com/en/comps/1/schedule/World-Cup-Scores-and-Fixtures"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

FBREF_NAME_MAP = {
    "United States":          "USA",
    "Côte d'Ivoire":          "Ivory Coast",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "DR Congo":               "DR Congo",
    "Cape Verde":             "Cape Verde",
    "Cape Verde Islands":     "Cape Verde",
    "Curacao":                "Curacao",
    "Curaçao":                "Curacao",
    "Korea Republic":         "South Korea",
}

def normalize_fbref_name(name):
    if not isinstance(name, str): return name
    return FBREF_NAME_MAP.get(name.strip(), name.strip())

print("Fetching xG data from FBRef...")
time.sleep(1)

XG_DATA = {}

try:
    resp = requests.get(FBREF_URL, headers=HEADERS, timeout=15)

    if resp.status_code != 200:
        raise Exception(f"HTTP {resp.status_code}")

    # Find all tables on the page and grab the one with xG columns
    tables = pd.read_html(resp.text)
    print(f"  Found {len(tables)} tables on page")

    schedule_df = None
    for t in tables:
        flat_cols = [str(c).lower() for c in t.columns.get_level_values(-1)]
        if sum("xg" in c for c in flat_cols) >= 2:
            schedule_df = t
            break

    if schedule_df is None:
        raise Exception("No table with xG columns found")

    # Flatten MultiIndex columns if present
    if isinstance(schedule_df.columns, pd.MultiIndex):
        schedule_df.columns = [
            f"{a}|{b}" if str(b) not in ("", str(a)) else str(a)
            for a, b in schedule_df.columns
        ]

    cols_lower   = {str(c).lower(): c for c in schedule_df.columns}
    home_col     = next((cols_lower[c] for c in cols_lower if "home" in c and "xg" not in c), None)
    away_col     = next((cols_lower[c] for c in cols_lower if "away" in c and "xg" not in c), None)
    xg_cols      = [cols_lower[c] for c in cols_lower if "xg" in c]

    if not home_col or not away_col or len(xg_cols) < 2:
        raise Exception(
            f"Column detection failed.\n"
            f"  home={home_col}, away={away_col}, xg_cols={xg_cols}\n"
            f"  All columns: {list(schedule_df.columns)}"
        )

    home_xg_col = xg_cols[0]
    away_xg_col = xg_cols[1]
    skipped     = 0

    for _, row in schedule_df.iterrows():
        home = normalize_fbref_name(row.get(home_col))
        away = normalize_fbref_name(row.get(away_col))
        if not isinstance(home, str) or not isinstance(away, str):
            skipped += 1
            continue
        try:
            hxg = float(row.get(home_xg_col))
            axg = float(row.get(away_xg_col))
            XG_DATA[(home, away)] = (hxg, axg)
        except (TypeError, ValueError):
            skipped += 1

    print(f"  ✓ {len(XG_DATA)} matches with xG | {skipped} skipped")
    print(f"\n  Sample xG values:")
    for (home, away), (hxg, axg) in list(XG_DATA.items())[:6]:
        print(f"    {home:<22} vs {away:<22}  {hxg:.1f} – {axg:.1f}  diff: {hxg-axg:+.1f}")

except Exception as e:
    print(f"  ✗ FBRef fetch failed: {e}")
    print("  → XG_DATA = {} — pipeline continues without xG blending")
    XG_DATA = {}

print(f"\n  Total matches with xG: {len(XG_DATA)}")

Fetching xG data from FBRef...
  ✗ FBRef fetch failed: HTTP 403
  → XG_DATA = {} — pipeline continues without xG blending

  Total matches with xG: 0


In [24]:
# ── WHERE THIS DATA COMES FROM ─────────────────────────────────────────────
# transfermarkt-datasets by dcaribou — 37,000+ players with market values
# and national team associations. Updated weekly.
# Hosted on Cloudflare R2. Fetched as a gzipped CSV, no API key needed.
#
# We group players by country_of_citizenship, take the top 26 by market
# value (WC squad size), and sum them. This gives us each nation's
# squad value as a data-derived starting Elo prior.

import gzip, io

PLAYERS_URL = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz"

# Transfermarkt country name → our standard name
TM_NAME_MAP = {
    "United States":                    "USA",
    "Côte d'Ivoire":                    "Ivory Coast",
    "Bosnia-Herzegovina":               "Bosnia and Herzegovina",
    "DR Congo":                         "DR Congo",
    "Democratic Republic of Congo":     "DR Congo",
    "Korea, South":                     "South Korea",
    "Cape Verde Islands":               "Cape Verde",
    "Curacao":                          "Curacao",
    "Curaçao":                          "Curacao",
}

def normalize_tm_name(name):
    if not isinstance(name, str): return name
    return TM_NAME_MAP.get(name.strip(), name.strip())

def fetch_squad_values(url, squad_size=26):
    """Fetch player data and decompress in memory."""
    print("Fetching player market values from Transfermarkt...")
    resp = requests.get(url, timeout=30)

    if resp.status_code != 200:
        raise Exception(f"Fetch failed: {resp.status_code}")

    compressed   = io.BytesIO(resp.content)
    decompressed = gzip.GzipFile(fileobj=compressed)
    df           = pd.read_csv(decompressed, low_memory=False)

    print(f"  ✓ {len(df):,} players loaded")
    return df

def compute_squad_values(players_df, squad_size=26):
    """
    For each country: take top 26 players by market value and sum them.
    Returns {team_name: total_squad_value_eur}
    """
    df = players_df[
        players_df['market_value_in_eur'].notna() &
        (players_df['market_value_in_eur'] > 0) &
        players_df['country_of_citizenship'].notna()
    ].copy()

    df['team'] = df['country_of_citizenship'].apply(normalize_tm_name)

    squad_values = (
        df.sort_values('market_value_in_eur', ascending=False)
          .groupby('team')
          .head(squad_size)
          .groupby('team')['market_value_in_eur']
          .sum()
          .to_dict()
    )
    return squad_values

# ── RUN IT ─────────────────────────────────────────────────────────────────
try:
    players_df   = fetch_squad_values(PLAYERS_URL)
    SQUAD_VALUES = compute_squad_values(players_df)

    print(f"\n  Squad values for {len(SQUAD_VALUES)} countries")
    print(f"\n  {'Team':<25} {'Value (€M)':>12}")
    print("  " + "─" * 39)
    for team, val in sorted(SQUAD_VALUES.items(), key=lambda x: -x[1])[:15]:
        print(f"  {team:<25} {val/1e6:>11.0f}M")

    print(f"\n  Sanity check:")
    for t in ["France", "Brazil", "England", "Argentina", "Morocco", "USA", "Qatar"]:
        val = SQUAD_VALUES.get(t, 0)
        print(f"    {t:<22} €{val/1e6:.0f}M")

except Exception as e:
    print(f"  Transfermarkt fetch failed: {e}")
    print("  Trying kagglehub fallback...")
    try:
        import kagglehub
        path         = kagglehub.dataset_download("davidcariboo/player-scores", path="players.csv")
        players_df   = pd.read_csv(path, low_memory=False)
        SQUAD_VALUES = compute_squad_values(players_df)
        print(f"  ✓ Loaded via kagglehub — {len(SQUAD_VALUES)} countries")
    except Exception as e2:
        print(f"  kagglehub also failed: {e2}")
        print("  SQUAD_VALUES set to empty — Cell 2B will use flat 1500 starts")
        SQUAD_VALUES = {}

Fetching player market values from Transfermarkt...
  ✓ 48,317 players loaded

  Squad values for 195 countries

  Team                        Value (€M)
  ───────────────────────────────────────
  France                           1843M
  England                          1780M
  Spain                            1535M
  Brazil                           1372M
  Germany                          1225M
  Portugal                         1195M
  Argentina                        1049M
  Netherlands                       985M
  Italy                             932M
  Belgium                           707M
  Norway                            640M
  Cote d'Ivoire                     612M
  Morocco                           611M
  Denmark                           545M
  Senegal                           532M

  Sanity check:
    France                 €1843M
    Brazil                 €1372M
    England                €1780M
    Argentina              €1049M
    Morocco                €611M
   

In [25]:
# ── MASTER NAME MAP ────────────────────────────────────────────────────────
# Converts any dataset's team name variants to our standard names.
# Used in all subsequent cells.
NAME_MAP = {
    "United States":                    "USA",
    "Democratic Republic of the Congo": "DR Congo",
    "Côte d'Ivoire":                    "Ivory Coast",
    "Bosnia-Herzegovina":               "Bosnia and Herzegovina",
    "Curacao":                          "Curacao",
    "Curaçao":                          "Curacao",
    "Cabo Verde":                       "Cape Verde",
    "Cape Verde Islands":               "Cape Verde",
    "Korea Republic":                   "South Korea",
}

def normalize_name(name):
    if not isinstance(name, str): return name
    return NAME_MAP.get(name.strip(), name.strip())

# ── TOURNAMENT WEIGHTS ─────────────────────────────────────────────────────
# World Cup games tell us more than friendlies.
# These weights scale how much each match type moves Elo ratings.
TOURNAMENT_WEIGHTS = {
    "FIFA World Cup":                        3.0,
    "Confederations Cup":                    2.5,
    "UEFA Euro":                             2.0,
    "Copa América":                          2.0,
    "Africa Cup of Nations":                 2.0,
    "AFC Asian Cup":                         2.0,
    "CONCACAF Gold Cup":                     2.0,
    "FIFA World Cup qualification":          2.0,
    "UEFA Euro qualification":               1.5,
    "Copa América qualification":            1.5,
    "Africa Cup of Nations qualification":   1.5,
    "AFC Asian Cup qualification":           1.5,
    "CONCACAF Gold Cup qualification":       1.5,
    "UEFA Nations League":                   1.5,
    "CONCACAF Nations League":               1.5,
    "AFC Asian Qualifiers":                  1.5,
    "Friendly":                              0.5,
}

def get_weight(tournament):
    if pd.isna(tournament): return 1.0
    t = str(tournament)
    if t in TOURNAMENT_WEIGHTS: return TOURNAMENT_WEIGHTS[t]
    for key, w in TOURNAMENT_WEIGHTS.items():
        if key.lower() in t.lower(): return w
    return 1.0

# ── PENALTY SHOOTOUT LOOKUP ────────────────────────────────────────────────
# These matches were level after 120 minutes — decided by a coin flip.
# Treating them as losses misrepresents the actual 90+30 minute performance.
SHOOTOUT_DRAWS = {
    ("Croatia",      "Brazil"):          "2022-12-09",
    ("Argentina",    "France"):          "2022-12-18",
    ("Argentina",    "Netherlands"):     "2022-12-09",
    ("Morocco",      "Spain"):           "2022-12-06",
    ("Morocco",      "Portugal"):        "2022-12-10",
    ("Japan",        "Croatia"):         "2022-12-05",
    ("Australia",    "Argentina"):       "2022-12-03",
    ("France",       "Switzerland"):     "2021-06-28",
    ("Spain",        "Switzerland"):     "2021-06-25",
    ("Colombia",     "Uruguay"):         "2021-07-03",
}

def is_shootout(home, away, date):
    d = str(date)[:10]
    return (
        d == SHOOTOUT_DRAWS.get((home, away)) or
        d == SHOOTOUT_DRAWS.get((away, home))
    )

# ── ELO MATH ───────────────────────────────────────────────────────────────
HOME_ADVANTAGE = 75

def expected_score(ra, rb):
    """Probability team A beats team B given their ratings."""
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def gd_multiplier(gd):
    """Bigger wins move ratings more. 4-0 matters more than 1-0."""
    if gd <= 1:   return 1.0
    elif gd == 2: return 1.5
    else:         return 1.75 + (gd - 3) * 0.1

# ── SQUAD VALUE PRIOR ──────────────────────────────────────────────────────
def squad_value_to_elo(value, squad_values_dict,
                        min_elo=1350, max_elo=1750):
    """
    Scale squad value to Elo range using actual min/max from the data.
    France €1350M → ~1750. Qatar €20M → ~1351.
    No hardcoded assumptions — the data determines the range.
    """
    all_vals   = [v for v in squad_values_dict.values() if v > 0]
    if not all_vals: return 1500.0
    min_val    = min(all_vals)
    max_val    = max(all_vals)
    if max_val == min_val: return 1500.0
    normalized = (value - min_val) / (max_val - min_val)
    return min_elo + normalized * (max_elo - min_elo)

# ── MAIN FUNCTION ──────────────────────────────────────────────────────────
def compute_base_elo(df, squad_values, start='2024-01-01', end='2026-06-10'):
    """
    Compute pre-tournament Elo ratings from real match history.

    Three improvements over a naive model:
    1. Squad value priors — teams start at ratings reflecting actual player quality
    2. Penalty shootouts treated as draws — 120 level minutes ≠ a loss
    3. 2024 window — recent form only, 2022 WC doesn't dominate
    4. Recency decay — last month matters more than 18 months ago
    """
    filtered = df[
        (df['date'] >= pd.Timestamp(start)) &
        (df['date'] <= pd.Timestamp(end))
    ].copy().sort_values('date')

    print(f"  Processing {len(filtered):,} matches ({start} → {end})")

    # Start from squad value priors instead of flat 1500
    elo = defaultdict(lambda: 1400.0)
    for team, value in squad_values.items():
        if value > 0:
            elo[team] = squad_value_to_elo(value, squad_values)

    print(f"\n  Starting priors from squad values:")
    for t in ["France", "Brazil", "England", "Argentina", "Morocco", "Norway", "USA"]:
        print(f"    {t:<22} {elo.get(t, 1400):.0f}")

    for _, row in filtered.iterrows():
        home = normalize_name(row['home_team'])
        away = normalize_name(row['away_team'])

        if pd.isna(row['home_score']) or pd.isna(row['away_score']):
            continue

        hg, ag = int(row['home_score']), int(row['away_score'])
        ra, rb = elo[home], elo[away]

        is_neutral   = row.get('neutral', False)
        ra_effective = ra if is_neutral else ra + HOME_ADVANTAGE
        ea           = expected_score(ra_effective, rb)

        # Penalty shootout → treat as draw
        if is_shootout(home, away, row['date']):
            sa = 0.5
        else:
            sa = 1.0 if hg > ag else (0.5 if hg == ag else 0.0)

        weight   = get_weight(row.get('tournament', ''))
        mult     = gd_multiplier(abs(hg - ag))
        days_ago = (pd.Timestamp('2026-06-10') - row['date']).days
        decay    = 0.997 ** days_ago
        k        = 32 * weight * mult * decay

        elo[home] = ra + k * (sa - ea)
        elo[away] = rb + k * ((1 - sa) - (1 - ea))

    return dict(elo)

# ── RUN IT ─────────────────────────────────────────────────────────────────
print("Computing BASE_ELO from historical data + squad value priors...\n")
BASE_ELO = compute_base_elo(historical_df, SQUAD_VALUES)

print(f"\n  Final ratings after match history applied:")
print(f"  {'Team':<25} {'Rating':>8}")
print("  " + "─" * 35)
for team, rating in sorted(BASE_ELO.items(), key=lambda x: -x[1])[:20]:
    print(f"  {team:<25} {rating:>8.0f}")

print(f"\n  Sanity check (should reflect real world order):")
for t in ["France", "Brazil", "England", "Argentina", "Spain",
          "Morocco", "Norway", "USA", "Qatar"]:
    print(f"    {t:<22} {BASE_ELO.get(t, 1400):.0f}")

Computing BASE_ELO from historical data + squad value priors...

  Processing 2,544 matches (2024-01-01 → 2026-06-10)

  Starting priors from squad values:
    France                 1750
    Brazil                 1648
    England                1736
    Argentina              1578
    Morocco                1483
    Norway                 1489
    USA                    1454

  Final ratings after match history applied:
  Team                        Rating
  ───────────────────────────────────
  England                       1756
  France                        1737
  Spain                         1731
  Germany                       1645
  Brazil                        1626
  Argentina                     1618
  Portugal                      1617
  Morocco                       1614
  Norway                        1607
  Netherlands                   1606
  Nigeria                       1580
  Italy                         1578
  Belgium                       1571
  Senegal         

In [26]:
def update_with_wc_results(wc_matches, base_elo, xg_data, alpha=0.4):
    """
    Update ratings using completed 2026 WC matches.
    Blends actual result with xG-implied result.

    alpha = 0.4 means:
      60% actual scoreline + 40% xG signal

    England 0-0 Ghana (xG 1.8 vs 0.4):
      actual_sa  = 0.5  (draw)
      xg_sa      = 1.0  (England dominated)
      blended_sa = 0.70  → England penalized less, Ghana credited less

    Brazil 3-0 Scotland (xG 3.1 vs 0.3):
      actual_sa  = 1.0
      xg_sa      = 1.0
      blended_sa = 1.0  → both signals agree, no change
    """
    elo      = dict(base_elo)
    done     = sorted([m for m in wc_matches if m.get("score")],
                      key=lambda m: m["date"])
    xg_used  = 0
    xg_miss  = 0

    for m in done:
        t1 = normalize_name(m["team1"])
        t2 = normalize_name(m["team2"])
        g1, g2 = m["score"]["ft"]
        r1 = elo.get(t1, 1500)
        r2 = elo.get(t2, 1500)

        # Actual result — shootouts treated as draws
        if is_shootout(t1, t2, m["date"]):
            actual_sa = 0.5
        else:
            actual_sa = 1.0 if g1 > g2 else (0.5 if g1 == g2 else 0.0)

        # xG signal — look up in both team orderings
        xg_pair    = xg_data.get((t1, t2))
        is_flipped = False
        if not xg_pair:
            xg_pair    = xg_data.get((t2, t1))
            is_flipped = True

        if xg_pair:
            hxg, axg = xg_pair
            if is_flipped:
                hxg, axg = axg, hxg

            # xG-implied result: gap >0.3 = one team meaningfully better
            if hxg > axg + 0.3:   xg_sa = 1.0
            elif axg > hxg + 0.3: xg_sa = 0.0
            else:                  xg_sa = 0.5

            blended_sa = (1 - alpha) * actual_sa + alpha * xg_sa
            xg_used += 1
        else:
            blended_sa = actual_sa
            xg_miss   += 1

        ea   = expected_score(r1, r2)
        mult = gd_multiplier(abs(g1 - g2))
        k    = 32 * mult

        elo[t1] = r1 + k * (blended_sa - ea)
        elo[t2] = r2 + k * ((1 - blended_sa) - (1 - ea))

    print(f"  ✓ Updated with {len(done)} WC matches")
    print(f"  xG blended: {xg_used} matches | Score only: {xg_miss} matches")
    return elo

# ── RUN IT ─────────────────────────────────────────────────────────────────
current_elo = update_with_wc_results(wc_matches, BASE_ELO, XG_DATA, alpha=0.4)

# Get all WC teams for display
wc_teams = set()
for m in wc_matches:
    wc_teams.add(normalize_name(m["team1"]))
    wc_teams.add(normalize_name(m["team2"]))

print(f"\n  Current ratings (BASE_ELO + WC results + xG):")
print(f"  {'Team':<25} {'Rating':>8}")
print("  " + "─" * 35)
top_wc = sorted(
    [(t, r) for t, r in current_elo.items() if t in wc_teams],
    key=lambda x: -x[1]
)
for team, rating in top_wc[:15]:
    print(f"  {team:<25} {rating:>8.0f}")

print(f"\n  Key teams after WC adjustment:")
for t in ["France", "Brazil", "England", "Argentina", "Spain",
          "Morocco", "USA", "Norway", "Qatar"]:
    print(f"    {t:<22} {current_elo.get(t, 1400):.0f}")

  ✓ Updated with 56 WC matches
  xG blended: 0 matches | Score only: 56 matches

  Current ratings (BASE_ELO + WC results + xG):
  Team                        Rating
  ───────────────────────────────────
  France                        1757
  England                       1755
  Spain                         1728
  Argentina                     1657
  Brazil                        1654
  Germany                       1646
  Morocco                       1635
  Norway                        1635
  Portugal                      1627
  Netherlands                   1623
  Belgium                       1561
  Switzerland                   1560
  Japan                         1546
  Mexico                        1542
  Colombia                      1539

  Key teams after WC adjustment:
    France                 1757
    Brazil                 1654
    England                1755
    Argentina              1657
    Spain                  1728
    Morocco                1635
    USA        

In [27]:
def simulate_group_match(t1, t2, elo):
    """
    Simulate one group stage match. Returns (goals_t1, goals_t2).
    Draw probability shrinks as the rating gap grows.
    """
    ra, rb = elo.get(t1, 1500), elo.get(t2, 1500)
    p_win  = expected_score(ra, rb)
    gap    = abs(ra - rb)
    p_draw = max(0.05, 0.22 - (gap / 400) * 0.10)

    r = random.random()
    if r < p_win * (1 - p_draw):
        return (random.randint(1, 3), 0)
    elif r < p_win * (1 - p_draw) + p_draw:
        g = random.randint(0, 2)
        return (g, g)
    else:
        return (0, random.randint(1, 3))

def simulate_ko_match(t1, t2, elo):
    """Knockout match — no draws. Returns winner."""
    return t1 if random.random() < expected_score(
        elo.get(t1, 1500), elo.get(t2, 1500)
    ) else t2

def get_standings_and_remaining(wc_matches):
    """
    Build current group standings from completed matches.
    Return remaining group matches to be simulated.
    """
    standings   = defaultdict(lambda: {"pts":0,"gf":0,"ga":0,"gd":0})
    group_teams = defaultdict(set)

    for m in wc_matches:
        grp = m.get("group", "")
        if not grp: continue

        t1 = normalize_name(m["team1"])
        t2 = normalize_name(m["team2"])
        group_teams[grp].update([t1, t2])

        if not m.get("score"): continue

        g1, g2 = m["score"]["ft"]

        standings[t1]["gf"] += g1; standings[t1]["ga"] += g2
        standings[t2]["gf"] += g2; standings[t2]["ga"] += g1
        standings[t1]["gd"] = standings[t1]["gf"] - standings[t1]["ga"]
        standings[t2]["gd"] = standings[t2]["gf"] - standings[t2]["ga"]

        if g1 > g2:    standings[t1]["pts"] += 3
        elif g1 == g2: standings[t1]["pts"] += 1; standings[t2]["pts"] += 1
        else:          standings[t2]["pts"] += 3

    remaining = [m for m in wc_matches if m.get("group") and not m.get("score")]
    return (
        dict(standings),
        {g: list(ts) for g, ts in group_teams.items()},
        remaining
    )

def simulate_tournament(elo, base_standings, group_teams, remaining):
    """
    Simulate one full tournament from current state.
    Returns the winning team name.

    Step 1: Simulate remaining group matches
    Step 2: Determine 32 qualifiers (top 2 per group + 8 best 3rd place)
    Step 3: Simulate knockout rounds until 1 team remains
    """
    standings = {t: dict(s) for t, s in base_standings.items()}

    # Step 1: Remaining group matches
    for m in remaining:
        t1 = normalize_name(m["team1"])
        t2 = normalize_name(m["team2"])
        g1, g2 = simulate_group_match(t1, t2, elo)

        for t, gf, ga in [(t1, g1, g2), (t2, g2, g1)]:
            if t not in standings:
                standings[t] = {"pts":0,"gf":0,"ga":0,"gd":0}
            standings[t]["gf"] += gf
            standings[t]["ga"] += ga
            standings[t]["gd"]  = standings[t]["gf"] - standings[t]["ga"]

        if g1 > g2:    standings[t1]["pts"] += 3
        elif g1 == g2: standings[t1]["pts"] += 1; standings[t2]["pts"] += 1
        else:          standings[t2]["pts"] += 3

    # Step 2: Determine qualifiers
    qualifiers  = []
    third_place = []

    for grp, members in sorted(group_teams.items()):
        ranked = sorted(members, key=lambda t: (
            -standings.get(t, {}).get("pts", 0),
            -standings.get(t, {}).get("gd",  0),
            -standings.get(t, {}).get("gf",  0)
        ))
        qualifiers += ranked[:2]
        if len(ranked) >= 3:
            third_place.append(ranked[2])

    third_place.sort(key=lambda t: (
        -standings.get(t, {}).get("pts", 0),
        -standings.get(t, {}).get("gd",  0)
    ))
    qualifiers += third_place[:8]

    # Step 3: Knockout rounds
    random.shuffle(qualifiers)
    alive = qualifiers[:]

    while len(alive) > 1:
        next_round = []
        for i in range(0, len(alive) - 1, 2):
            next_round.append(simulate_ko_match(alive[i], alive[i+1], elo))
        if len(alive) % 2 == 1:
            next_round.append(alive[-1])
        alive = next_round

    return alive[0] if alive else None

def run_simulation(wc_matches, elo, n=10000):
    """Run n simulations. Returns {team: win_probability}."""
    print(f"Running {n:,} simulations...")
    standings, group_teams, remaining = get_standings_and_remaining(wc_matches)
    print(f"  Group matches still to play: {len(remaining)}")

    win_counts = defaultdict(int)
    for _ in range(n):
        winner = simulate_tournament(elo, standings, group_teams, remaining)
        if winner: win_counts[winner] += 1

    win_probs = {t: c / n for t, c in win_counts.items()}
    print(f"  Done. Probabilities sum to: {sum(win_probs.values()):.3f}\n")
    return win_probs

# ── RUN IT ─────────────────────────────────────────────────────────────────
win_probs = run_simulation(wc_matches, current_elo, n=10000)

print(f"{'Team':<25} {'Win Prob':>10}")
print("─" * 37)
for team, prob in sorted(win_probs.items(), key=lambda x: -x[1])[:15]:
    print(f"{team:<25} {prob:>10.1%}")

Running 10,000 simulations...
  Group matches still to play: 16
  Done. Probabilities sum to: 1.000

Team                        Win Prob
─────────────────────────────────────
France                         18.8%
England                        17.5%
Spain                          13.9%
Argentina                       6.3%
Brazil                          5.9%
Germany                         5.6%
Norway                          4.7%
Morocco                         4.6%
Portugal                        4.1%
Netherlands                     3.8%
Switzerland                     1.6%
Mexico                          1.4%
Belgium                         1.3%
Japan                           1.2%
Colombia                        1.1%


In [28]:
# ── MARKET PROBABILITIES ───────────────────────────────────────────────────
# Implied win probabilities from Kalshi/FanDuel as of June 24, 2026.
# A decimal of 0.201 = 20.1% chance of winning the tournament.
# Converted from American odds: France +360 → 1/(1+3.60) = 21.7%
# When we move to VS Code this becomes a live Polymarket API call.
MARKET_PROBS = {
    "France":      0.217,
    "Spain":       0.154,
    "England":     0.143,
    "Argentina":   0.133,
    "Portugal":    0.091,
    "Brazil":      0.071,
    "Germany":     0.071,
    "Netherlands": 0.056,
    "USA":         0.032,
    "Norway":      0.032,
    "Morocco":     0.028,
    "Japan":       0.022,
    "Mexico":      0.022,
    "Colombia":    0.022,
}

def signal(delta):
    """
    Flag meaningful divergences between model and market.
    delta = model probability minus market probability.
    Positive = model more bullish. Negative = market more bullish.
    Only flag gaps larger than 4% — smaller gaps are noise.
    """
    if delta >  0.04: return "🟢 MODEL FAVORS"
    if delta < -0.04: return "🔴 MARKET FAVORS"
    return "⚪ neutral"

# ── BUILD COMPARISON TABLE ─────────────────────────────────────────────────
rows = []
for team, model_p in win_probs.items():
    if model_p < 0.002: continue
    market_p = MARKET_PROBS.get(team)
    delta    = (model_p - market_p) if market_p is not None else None
    rows.append((team, model_p, market_p, delta))

rows.sort(key=lambda x: -x[1])

print(f"{'Team':<22} {'Elo':>6} {'Model':>7} {'Market':>7} {'Delta':>7}  Signal")
print("─" * 68)

for team, model_p, market_p, delta in rows:
    elo_r = current_elo.get(team, 0)
    m_str = f"{market_p:.1%}" if market_p  is not None else "  N/A"
    d_str = f"{delta:+.1%}"   if delta     is not None else "  N/A"
    sig   = signal(delta)     if delta     is not None else ""
    print(f"{team:<22} {elo_r:>6.0f} {model_p:>7.1%} {m_str:>7} {d_str:>7}  {sig}")

# ── THE STORY ──────────────────────────────────────────────────────────────
# Every divergence is a conversation:
# "Here's what my model sees. Here's what the market sees.
#  Here's what each one is missing."
print(f"\n── BIGGEST DIVERGENCES ──")
divergences = [
    (t, mp, mkt, d) for t, mp, mkt, d in rows
    if d is not None and abs(d) >= 0.03
]
for team, model_p, market_p, delta in sorted(divergences, key=lambda x: -abs(x[3])):
    direction = "model sees more upside" if delta > 0 else "market more bullish"
    print(f"  {team:<20}  model {model_p:.1%}  market {market_p:.1%}  →  {direction}")

Team                      Elo   Model  Market   Delta  Signal
────────────────────────────────────────────────────────────────────
France                   1757   18.8%   21.7%   -2.9%  ⚪ neutral
England                  1755   17.5%   14.3%   +3.2%  ⚪ neutral
Spain                    1728   13.9%   15.4%   -1.5%  ⚪ neutral
Argentina                1657    6.3%   13.3%   -7.0%  🔴 MARKET FAVORS
Brazil                   1654    5.9%    7.1%   -1.2%  ⚪ neutral
Germany                  1646    5.6%    7.1%   -1.5%  ⚪ neutral
Norway                   1635    4.7%    3.2%   +1.5%  ⚪ neutral
Morocco                  1635    4.6%    2.8%   +1.8%  ⚪ neutral
Portugal                 1627    4.1%    9.1%   -5.0%  🔴 MARKET FAVORS
Netherlands              1623    3.8%    5.6%   -1.8%  ⚪ neutral
Switzerland              1560    1.6%     N/A     N/A  
Mexico                   1542    1.4%    2.2%   -0.8%  ⚪ neutral
Belgium                  1561    1.3%     N/A     N/A  
Japan                    1546 

In [29]:
# ── CELL 6: DESCRIPTIVE STATS + FUN FACTS ─────────────────────────────────
# Requires: wc_matches, current_elo, BASE_ELO, win_probs, XG_DATA
# All reference data is inline — no external files needed.

import json
from collections import defaultdict

# ── QUIRKY REFERENCE DATA (inline — no file needed) ───────────────────────
# Jersey colors, populations, coach info, WC history for all 48 teams.
# Moves to quirky_data.json when we build the VS Code pipeline.
QUIRKY_TEAMS = {
    "France":       {"continent":"Europe",       "jersey":"blue",             "wc_wins":2, "population_millions":68.4,  "coach":"Didier Deschamps",   "coach_nat":"French"},
    "Brazil":       {"continent":"South America","jersey":"yellow",           "wc_wins":5, "population_millions":215.3, "coach":"Dorival Junior",     "coach_nat":"Brazilian"},
    "Argentina":    {"continent":"South America","jersey":"blue_white",       "wc_wins":3, "population_millions":46.2,  "coach":"Lionel Scaloni",     "coach_nat":"Argentine"},
    "Spain":        {"continent":"Europe",       "jersey":"red",              "wc_wins":1, "population_millions":47.4,  "coach":"Luis de la Fuente",  "coach_nat":"Spanish"},
    "England":      {"continent":"Europe",       "jersey":"white",            "wc_wins":1, "population_millions":56.5,  "coach":"Thomas Tuchel",      "coach_nat":"German"},
    "Germany":      {"continent":"Europe",       "jersey":"white",            "wc_wins":4, "population_millions":84.1,  "coach":"Julian Nagelsmann",  "coach_nat":"German"},
    "Portugal":     {"continent":"Europe",       "jersey":"red",              "wc_wins":0, "population_millions":10.3,  "coach":"Roberto Martinez",   "coach_nat":"Spanish"},
    "Netherlands":  {"continent":"Europe",       "jersey":"orange",           "wc_wins":0, "population_millions":17.9,  "coach":"Ronald Koeman",      "coach_nat":"Dutch"},
    "Belgium":      {"continent":"Europe",       "jersey":"red",              "wc_wins":0, "population_millions":11.6,  "coach":"Domenico Tedesco",   "coach_nat":"Italian"},
    "Colombia":     {"continent":"South America","jersey":"yellow",           "wc_wins":0, "population_millions":51.9,  "coach":"Néstor Lorenzo",     "coach_nat":"Argentine"},
    "Uruguay":      {"continent":"South America","jersey":"sky_blue",         "wc_wins":2, "population_millions":3.5,   "coach":"Marcelo Bielsa",     "coach_nat":"Argentine"},
    "Japan":        {"continent":"Asia",         "jersey":"blue",             "wc_wins":0, "population_millions":125.7, "coach":"Hajime Moriyasu",    "coach_nat":"Japanese"},
    "Morocco":      {"continent":"Africa",       "jersey":"red",              "wc_wins":0, "population_millions":37.5,  "coach":"Walid Regragui",     "coach_nat":"Moroccan"},
    "USA":          {"continent":"North America","jersey":"white",            "wc_wins":0, "population_millions":335.9, "coach":"Mauricio Pochettino","coach_nat":"Argentine"},
    "Mexico":       {"continent":"North America","jersey":"green",            "wc_wins":0, "population_millions":130.2, "coach":"Javier Aguirre",     "coach_nat":"Mexican"},
    "Norway":       {"continent":"Europe",       "jersey":"red",              "wc_wins":0, "population_millions":5.5,   "coach":"Ståle Solbakken",    "coach_nat":"Norwegian"},
    "Sweden":       {"continent":"Europe",       "jersey":"yellow",           "wc_wins":0, "population_millions":10.5,  "coach":"Jon Dahl Tomasson",  "coach_nat":"Danish"},
    "Australia":    {"continent":"Oceania",      "jersey":"yellow",           "wc_wins":0, "population_millions":26.5,  "coach":"Tony Popovic",       "coach_nat":"Australian"},
    "South Korea":  {"continent":"Asia",         "jersey":"red",              "wc_wins":0, "population_millions":51.7,  "coach":"Hong Myung-bo",      "coach_nat":"South Korean"},
    "Ecuador":      {"continent":"South America","jersey":"yellow",           "wc_wins":0, "population_millions":18.0,  "coach":"Sebastián Beccacece","coach_nat":"Argentine"},
    "Senegal":      {"continent":"Africa",       "jersey":"white",            "wc_wins":0, "population_millions":17.9,  "coach":"Aliou Cissé",        "coach_nat":"Senegalese"},
    "Ghana":        {"continent":"Africa",       "jersey":"white",            "wc_wins":0, "population_millions":33.5,  "coach":"Carlos Queiroz",     "coach_nat":"Portuguese"},
    "Croatia":      {"continent":"Europe",       "jersey":"red_white_checks", "wc_wins":0, "population_millions":3.9,   "coach":"Zlatko Dalić",       "coach_nat":"Croatian"},
    "Canada":       {"continent":"North America","jersey":"red",              "wc_wins":0, "population_millions":38.9,  "coach":"Jesse Marsch",       "coach_nat":"American"},
    "Switzerland":  {"continent":"Europe",       "jersey":"red",              "wc_wins":0, "population_millions":8.7,   "coach":"Murat Yakin",        "coach_nat":"Swiss"},
    "Austria":      {"continent":"Europe",       "jersey":"red",              "wc_wins":0, "population_millions":9.1,   "coach":"Ralf Rangnick",      "coach_nat":"German"},
    "Algeria":      {"continent":"Africa",       "jersey":"white",            "wc_wins":0, "population_millions":45.6,  "coach":"Vladimir Petkovic",  "coach_nat":"Swiss"},
    "Turkey":       {"continent":"Europe",       "jersey":"red",              "wc_wins":0, "population_millions":85.3,  "coach":"Vincenzo Montella",  "coach_nat":"Italian"},
    "Scotland":     {"continent":"Europe",       "jersey":"blue",             "wc_wins":0, "population_millions":5.5,   "coach":"Steve Clarke",       "coach_nat":"Scottish"},
    "Paraguay":     {"continent":"South America","jersey":"red_white",        "wc_wins":0, "population_millions":7.4,   "coach":"Gustavo Alfaro",     "coach_nat":"Argentine"},
    "Saudi Arabia": {"continent":"Asia",         "jersey":"green",            "wc_wins":0, "population_millions":36.4,  "coach":"Roberto Mancini",    "coach_nat":"Italian"},
    "Cape Verde":   {"continent":"Africa",       "jersey":"blue",             "wc_wins":0, "population_millions":0.6,   "coach":"Pedro Leitão Brito", "coach_nat":"Cape Verdean"},
    "Cabo Verde":   {"continent":"Africa",       "jersey":"blue",             "wc_wins":0, "population_millions":0.6,   "coach":"Pedro Leitão Brito", "coach_nat":"Cape Verdean"},
    "Tunisia":      {"continent":"Africa",       "jersey":"red",              "wc_wins":0, "population_millions":12.0,  "coach":"Jalel Kadri",        "coach_nat":"Tunisian"},
    "Ivory Coast":  {"continent":"Africa",       "jersey":"orange",           "wc_wins":0, "population_millions":27.5,  "coach":"Emerse Faé",         "coach_nat":"Ivorian"},
    "Egypt":        {"continent":"Africa",       "jersey":"red",              "wc_wins":0, "population_millions":105.9, "coach":"Hossam Hassan",      "coach_nat":"Egyptian"},
    "Iran":         {"continent":"Asia",         "jersey":"white",            "wc_wins":0, "population_millions":87.9,  "coach":"Amir Ghalenoei",     "coach_nat":"Iranian"},
    "New Zealand":  {"continent":"Oceania",      "jersey":"white",            "wc_wins":0, "population_millions":5.1,   "coach":"Darren Bazeley",     "coach_nat":"New Zealander"},
    "DR Congo":     {"continent":"Africa",       "jersey":"blue",             "wc_wins":0, "population_millions":102.3, "coach":"Sébastien Desabre",  "coach_nat":"French"},
    "Bosnia and Herzegovina": {"continent":"Europe","jersey":"blue",          "wc_wins":0, "population_millions":3.3,   "coach":"Sergej Barbarez",    "coach_nat":"Bosnian"},
    "Qatar":        {"continent":"Asia",         "jersey":"maroon",           "wc_wins":0, "population_millions":2.9,   "coach":"Marquez Lopez",      "coach_nat":"Spanish"},
    "Iraq":         {"continent":"Asia",         "jersey":"green",            "wc_wins":0, "population_millions":43.5,  "coach":"Jesus Casas",        "coach_nat":"Spanish"},
    "Jordan":       {"continent":"Asia",         "jersey":"white",            "wc_wins":0, "population_millions":10.3,  "coach":"Hussein Ammouta",    "coach_nat":"Moroccan"},
    "Haiti":        {"continent":"North America","jersey":"blue",             "wc_wins":0, "population_millions":11.7,  "coach":"Marc Collat",        "coach_nat":"French"},
    "Panama":       {"continent":"North America","jersey":"red",              "wc_wins":0, "population_millions":4.4,   "coach":"Thomas Christiansen","coach_nat":"Danish"},
    "South Africa": {"continent":"Africa",       "jersey":"yellow",           "wc_wins":0, "population_millions":60.4,  "coach":"Hugo Broos",         "coach_nat":"Belgian"},
    "Czech Republic":{"continent":"Europe",      "jersey":"red",              "wc_wins":0, "population_millions":10.9,  "coach":"Ivan Hasek",         "coach_nat":"Czech"},
    "Curacao":      {"continent":"North America","jersey":"blue",             "wc_wins":0, "population_millions":0.19,  "coach":"Remko Bicentini",    "coach_nat":"Dutch"},
    "Curaçao":      {"continent":"North America","jersey":"blue",             "wc_wins":0, "population_millions":0.19,  "coach":"Remko Bicentini",    "coach_nat":"Dutch"},
    "Uzbekistan":   {"continent":"Asia",         "jersey":"white",            "wc_wins":0, "population_millions":36.5,  "coach":"Srecko Katanec",     "coach_nat":"Slovenian"},
}

# Historical World Cup wins grouped by jersey color
COLOR_WINS = {
    "yellow":          {"wins":5, "winners":["Brazil x5"]},
    "white":           {"wins":5, "winners":["Germany x4","England x1"]},
    "blue":            {"wins":4, "winners":["France x2","Uruguay x2"]},
    "blue_white":      {"wins":3, "winners":["Argentina x3"]},
    "red":             {"wins":1, "winners":["Spain x1"]},
    "orange":          {"wins":0, "winners":[]},
    "green":           {"wins":0, "winners":[]},
    "sky_blue":        {"wins":0, "winners":[]},
    "red_white_checks":{"wins":0, "winners":[]},
    "maroon":          {"wins":0, "winners":[]},
}

wc_teams_set = set(
    [m["team1"] for m in wc_matches] +
    [m["team2"] for m in wc_matches]
)

# ── 1. GOLDEN BOOT ─────────────────────────────────────────────────────────
scorers = defaultdict(lambda: {"goals":0,"team":"","minutes":[],"matches":set()})
for m in wc_matches:
    if not m.get("score"): continue
    for side, team in [("goals1", m["team1"]), ("goals2", m["team2"])]:
        for goal in m.get(side, []):
            if goal.get("owngoal"): continue
            name = goal.get("name","Unknown")
            scorers[name]["goals"]   += 1
            scorers[name]["team"]     = team
            scorers[name]["minutes"].append(goal.get("minute","?"))
            scorers[name]["matches"].add(m.get("date",""))

golden_boot = sorted([
    {"player":p,"team":d["team"],"goals":d["goals"],
     "matches_scored_in":len(d["matches"]),"minutes":d["minutes"]}
    for p,d in scorers.items()
], key=lambda x: -x["goals"])

print("═" * 55)
print("  GOLDEN BOOT RACE")
print("═" * 55)
print(f"  {'Player':<25} {'Team':<15} {'Goals':>5}")
print("  " + "─" * 50)
for p in golden_boot[:10]:
    print(f"  {p['player']:<25} {p['team']:<15} {p['goals']:>5}")

# ── 2. MATCH EXCITEMENT INDEX ──────────────────────────────────────────────
excitement = []
for m in wc_matches:
    if not m.get("score"): continue
    g1,g2 = m["score"]["ft"]
    t1,t2 = m["team1"],m["team2"]
    xg_total = None
    if XG_DATA:
        pair = XG_DATA.get((t1,t2)) or XG_DATA.get((t2,t1))
        if pair: xg_total = round(pair[0]+pair[1], 2)
    excitement.append({
        "match":f"{t1} vs {t2}","score":f"{g1}-{g2}",
        "total_goals":g1+g2,"xg_total":xg_total,
        "excitement":round((g1+g2)+(xg_total*0.3 if xg_total else 0), 2)
    })
excitement.sort(key=lambda x: -x["excitement"])

print(f"\n{'═'*55}")
print("  MOST EXCITING MATCHES")
print("═" * 55)
print(f"  {'Match':<38} {'Score':<6} {'xG':>6}")
print("  " + "─" * 54)
for m in excitement[:8]:
    xg = f"{m['xg_total']:.1f}" if m['xg_total'] else "N/A"
    print(f"  {m['match']:<38} {m['score']:<6} {xg:>6}")

# ── 3. xG EFFICIENCY ───────────────────────────────────────────────────────
team_goals = defaultdict(int)
team_xgf   = defaultdict(float)
for m in wc_matches:
    if not m.get("score"): continue
    t1,t2=m["team1"],m["team2"]; g1,g2=m["score"]["ft"]
    team_goals[t1]+=g1; team_goals[t2]+=g2
    if XG_DATA:
        pair=XG_DATA.get((t1,t2)) or XG_DATA.get((t2,t1))
        if pair:
            hxg,axg=pair if XG_DATA.get((t1,t2)) else (pair[1],pair[0])
            team_xgf[t1]+=hxg; team_xgf[t2]+=axg

xg_efficiency = []
for team in set(list(team_goals)+list(team_xgf)):
    goals=team_goals.get(team,0); xgf=team_xgf.get(team,0)
    eff=round(goals/xgf,2) if xgf>0 else None
    label=("🟢 clinical" if eff and eff>1.15 else
           "🔴 wasteful" if eff and eff<0.85 else "⚪ on track")
    xg_efficiency.append({"team":team,"goals":goals,
                           "xg_for":round(xgf,2),"efficiency":eff,"label":label})
xg_efficiency.sort(key=lambda x: -(x["efficiency"] or 0))

print(f"\n{'═'*55}")
print("  xG EFFICIENCY (clinical vs wasteful)")
print("═" * 55)
print(f"  {'Team':<22} {'Goals':>5} {'xG For':>7} {'Ratio':>7}  Status")
print("  " + "─" * 54)
for t in xg_efficiency:
    if t["xg_for"] > 0:
        eff_str = f"{t['efficiency']:.2f}" if t['efficiency'] else "N/A"
        print(f"  {t['team']:<22} {t['goals']:>5} {t['xg_for']:>7.1f} {eff_str:>7}  {t['label']}")

# ── 4. CLEAN SHEET TRACKER ─────────────────────────────────────────────────
cs=defaultdict(int); games=defaultdict(int)
for m in wc_matches:
    if not m.get("score"): continue
    t1,t2=m["team1"],m["team2"]; g1,g2=m["score"]["ft"]
    games[t1]+=1; games[t2]+=1
    if g2==0: cs[t1]+=1
    if g1==0: cs[t2]+=1

clean_sheets = [{"team":t,"clean_sheets":cs.get(t,0),
                 "games":games[t],"rate":round(cs.get(t,0)/games[t],2)}
                for t in games]
clean_sheets.sort(key=lambda x: (-x["clean_sheets"],-x["rate"]))

print(f"\n{'═'*55}")
print("  CLEAN SHEET TRACKER")
print("═" * 55)
for t in clean_sheets[:8]:
    shields = "🛡️" * t["clean_sheets"]
    print(f"  {t['team']:<22} {t['clean_sheets']}/{t['games']} games  {shields}")

# ── 5. GOAL TIMING HISTOGRAM ───────────────────────────────────────────────
bins = {"0-15":0,"16-30":0,"31-45":0,"46-60":0,"61-75":0,"76-90":0,"90+":0}
for m in wc_matches:
    if not m.get("score"): continue
    for side in ["goals1","goals2"]:
        for goal in m.get(side,[]):
            if goal.get("owngoal"): continue
            try:
                ms = str(goal.get("minute","0"))
                minute = 90 if "+" in ms else int(ms.replace("'","").strip())
                if minute<=15:   bins["0-15"]+=1
                elif minute<=30: bins["16-30"]+=1
                elif minute<=45: bins["31-45"]+=1
                elif minute<=60: bins["46-60"]+=1
                elif minute<=75: bins["61-75"]+=1
                elif minute<=90: bins["76-90"]+=1
                else:            bins["90+"]+=1
            except: pass

print(f"\n{'═'*55}")
print("  WHEN ARE GOALS BEING SCORED?")
print("═" * 55)
total_goals = sum(bins.values())
for period, count in bins.items():
    bar = "█" * count
    print(f"  {period:<8} {bar:<30} {count}")
print(f"  Total goals so far: {total_goals}")

# ── 6. BIGGEST UPSETS ──────────────────────────────────────────────────────
upsets = []
for m in wc_matches:
    if not m.get("score"): continue
    t1,t2=m["team1"],m["team2"]; g1,g2=m["score"]["ft"]
    r1=BASE_ELO.get(t1,1500); r2=BASE_ELO.get(t2,1500)
    p1=1/(1+10**((r2-r1)/400))
    if g1>g2:   result=f"{t1} win";  surprise=1-p1
    elif g2>g1: result=f"{t2} win";  surprise=p1
    else:       result="draw";       surprise=abs(p1-0.5)*0.7
    upsets.append({"match":f"{t1} vs {t2}","score":f"{g1}-{g2}",
                   "result":result,"t1_prob":round(p1,2),
                   "surprise":round(surprise,3)})
upsets.sort(key=lambda x: -x["surprise"])

print(f"\n{'═'*55}")
print("  BIGGEST UPSETS BY PROBABILITY")
print("═" * 55)
print(f"  {'Match':<38} {'Score':<6} {'Surprise':>8}")
print("  " + "─" * 55)
for u in upsets[:8]:
    print(f"  {u['match']:<38} {u['score']:<6} {u['surprise']:>8.3f}")

# ── 7. ELO MOVERS ──────────────────────────────────────────────────────────
movers = [{"team":t,
            "pre":round(BASE_ELO.get(t,1500)),
            "current":round(current_elo.get(t,1500)),
            "change":round(current_elo.get(t,1500)-BASE_ELO.get(t,1500))}
          for t in wc_teams_set]
movers.sort(key=lambda x: -x["change"])

print(f"\n{'═'*55}")
print("  ELO MOVERS — rising and falling")
print("═" * 55)
print(f"  {'Team':<22} {'Pre-WC':>7} {'Now':>7} {'Change':>8}")
print("  " + "─" * 48)
for m in movers[:10]:
    arrow = "↑" if m["change"]>0 else "↓" if m["change"]<0 else "→"
    print(f"  {m['team']:<22} {m['pre']:>7} {m['current']:>7} {m['change']:>+8}  {arrow}")

# ── 8. OPPONENT STRENGTH ───────────────────────────────────────────────────
opp_data = defaultdict(list)
for m in wc_matches:
    if not m.get("score"): continue
    t1,t2=m["team1"],m["team2"]
    opp_data[t1].append(current_elo.get(t2,1500))
    opp_data[t2].append(current_elo.get(t1,1500))

opp_strength = [{"team":t,"avg_opp_elo":round(sum(v)/len(v)),"games":len(v)}
                for t,v in opp_data.items()]
opp_strength.sort(key=lambda x: -x["avg_opp_elo"])

print(f"\n{'═'*55}")
print("  OPPONENT STRENGTH (hardest path so far)")
print("═" * 55)
print(f"  {'Team':<22} {'Avg Opp Elo':>12} {'Games':>6}")
print("  " + "─" * 43)
for t in opp_strength[:10]:
    print(f"  {t['team']:<22} {t['avg_opp_elo']:>12} {t['games']:>6}")

# ── 9. FUN FACTS + QUIRKY ANGLES ──────────────────────────────────────────
print(f"\n{'═'*55}")
print("  FUN FACTS + QUIRKY ANGLES")
print("═" * 55)

# Jersey color wins
print("\n  Jersey color — World Cup wins by home kit:")
for color, data in sorted(COLOR_WINS.items(), key=lambda x: -x[1]["wins"]):
    if data["wins"] > 0:
        winners = ", ".join(data["winners"])
        print(f"    {color:<25} {data['wins']} wins  ({winners})")

# Smallest nations still in the tournament
print("\n  Smallest nations still in the tournament:")
for team in sorted(wc_teams_set,
    key=lambda t: QUIRKY_TEAMS.get(t,{}).get("population_millions",999)):
    info = QUIRKY_TEAMS.get(team,{})
    pop  = info.get("population_millions")
    if pop and pop < 15:
        prob = win_probs.get(team,0)
        print(f"    {team:<22} pop: {pop}M   win prob: {prob:.1%}")

# Foreign coaches
print("\n  Teams with foreign coaches:")
for team in sorted(wc_teams_set):
    info  = QUIRKY_TEAMS.get(team,{})
    coach = info.get("coach","")
    nat   = info.get("coach_nat","")
    if nat and nat.lower() not in team.lower():
        print(f"    {team:<22} {coach} ({nat})")

# Historical pattern
print("\n  📊 The big historical pattern:")
print("  Every World Cup since 1978 won by South America or Europe.")
print("  12 consecutive tournaments. 0 exceptions.")
print("  In 2026 the host is North America. Does the pattern hold?")
print(f"\n  Current model favorites:")
for team, prob in sorted(win_probs.items(), key=lambda x: -x[1])[:5]:
    continent = QUIRKY_TEAMS.get(team,{}).get("continent","?")
    print(f"    {team:<22} {prob:.1%}  ({continent})")

print(f"\n  ✓ Cell 6 complete")

═══════════════════════════════════════════════════════
  GOLDEN BOOT RACE
═══════════════════════════════════════════════════════
  Player                    Team            Goals
  ──────────────────────────────────────────────────
  Lionel Messi              Argentina           5
  Vinícius Júnior           Brazil              4
  Kylian Mbappé             France              4
  Erling Haaland            Norway              4
  Johan Manzambi            Switzerland         3
  Jonathan David            Canada              3
  Ismael Saibari            Morocco             3
  Matheus Cunha             Brazil              3
  Deniz Undav               Germany             3
  Julián Quiñones           Mexico              2

═══════════════════════════════════════════════════════
  MOST EXCITING MATCHES
═══════════════════════════════════════════════════════
  Match                                  Score      xG
  ──────────────────────────────────────────────────────
  Germany vs Cura